# Árvore de Decisão

**Sessão 2 · Classificação · Parte 1**

*Inteligência Artificial e Aprendizagem de Máquina · FECAP · 2026/02*

## Carregando os animais e treinando uma árvore pequena

Mesmos 85 animais, mesmos atributos booleanos do Jogo do Bicho Misterioso (Aula 01) — agora deixamos o algoritmo escolher as perguntas sozinho.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

animais = json.load(open("animais_completo.json", encoding="utf-8"))
df = pd.DataFrame(animais)

features = ["voa", "nada", "bota_ovos", "tem_quatro_patas",
            "vive_na_agua", "e_carnivoro", "tem_cauda", "e_domestico"]
X = df[features]
y = df["tem_pelos"]

modelo = DecisionTreeClassifier(max_depth=2, random_state=42)
modelo.fit(X, y)

**Visualizando a árvore que o algoritmo aprendeu:**

In [ ]:
plt.figure(figsize=(14,7))
plot_tree(modelo, feature_names=features,
          class_names=["sem pelo", "com pelo"],
          filled=True, rounded=True)
plt.show()

## Os erros da árvore

Ela erra em 4 animais: porco, aranha, abelha e elefante — casos de fronteira genuínos (porco tem pelo esparso; aranha e abelha são "peludas" mas não são mamíferos; elefante tem pelo mínimo).

In [ ]:
pred = modelo.predict(X)
erros = df[pred != y][['nome']]
print('Animais que a arvore errou:')
print(erros)

## Como o algoritmo decide onde perguntar

A cada divisão possível, o algoritmo mede a **impureza de Gini** — o quão "misturadas" as classes ficam depois da pergunta. Ele escolhe a pergunta que mais separa bem as classes.

Mesmo papel que o erro quadrático teve na regressão, e o log loss na regressão logística.

## O papel da profundidade (`max_depth`)

Muito rasa: erra por simplicidade demais. Muito funda: decora o treino, generaliza mal.

**Testando várias profundidades:**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5, stratify=y
)

profundidades = range(1, 11)
acc_treino, acc_teste = [], []
for d in profundidades:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    acc_treino.append(accuracy_score(y_train, m.predict(X_train)))
    acc_teste.append(accuracy_score(y_test, m.predict(X_test)))

plt.plot(list(profundidades), acc_treino, marker="o", label="Treino")
plt.plot(list(profundidades), acc_teste, marker="o", label="Teste")
plt.xlabel("max_depth")
plt.ylabel("Acuracia")
plt.legend()
plt.show()

## O fluxo no Scikit-Learn

Mesma gramática de sempre: `DecisionTreeClassifier(max_depth=D)`, `fit()`, `predict()`.

In [ ]:
modelo_final = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo_final.fit(X_train, y_train)

pred = modelo_final.predict(X_test)
print(f"Acuracia: {accuracy_score(y_test, pred):.2%}")

## Importância das features

`feature_importances_` mostra quais variáveis mais pesaram nas decisões da árvore inteira — um bônus que só árvore (e Random Forest) têm.

In [ ]:
importancias = pd.Series(
    modelo_final.feature_importances_, index=features
).sort_values()

importancias.plot(kind="barh")
plt.xlabel("Importancia")
plt.show()

---
## Fechando Árvore de Decisão

Vimos como o algoritmo aprende sozinho quais perguntas fazer (Gini), o equilíbrio da profundidade, e como interpretar quais features mais importam.